<a href="https://colab.research.google.com/github/Ragul-S-2025/ML/blob/main/gridcv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np

In [29]:
data=pd.read_csv("/content/Social_Network_Ads.csv")
data.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [30]:
data.tail()

,User ID,Gender,Age,EstimatedSalary,Purchased
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0
399,15594041,Female,49,36000,1


In [31]:
data=pd.get_dummies(data,dtype=int,drop_first=True)
data.head()

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1


In [32]:
X=data[["User ID","Age","Gender_Male","Purchased"]]
X

,User ID,Age,Gender_Male,Purchased
0,15624510,19,1,0
1,15810944,35,1,0
2,15668575,26,0,0
3,15603246,27,0,0
4,15804002,19,1,0
...,...,...,...,...
395,15691863,46,0,1
396,15706071,51,1,1
397,15654296,50,0,1
398,15755018,36,1,0


In [33]:
y=data[["EstimatedSalary"]]
y

,EstimatedSalary
0,19000
1,20000
2,43000
3,57000
4,76000
...,...
395,41000
396,23000
397,20000
398,33000


In [34]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = data[["Age","Gender_Male","Purchased"]] # Exclude 'User ID' from features
X = sc.fit_transform(X)

In [35]:
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier # Import KNeighborsClassifier

# Define parameter grid with valid parameters for KNeighborsClassifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2] # For minkowski metric, p=1 is manhattan_distance, p=2 is euclidean_distance
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 5,n_jobs=-1,scoring='f1_weighted')

# fitting the model for grid search
grid.fit(X,y.values.ravel())

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


GridSearchCV(estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan', 'minkowski'],
                         'n_neighbors': [3, 5, 7, 9], 'p': [1, 2],
                         'weights': ['uniform', 'distance']},
             scoring='f1_weighted', verbose=5)

In [36]:
y_pred = cross_val_predict(grid.best_estimator_, X, y, cv=3)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [37]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
cm = confusion_matrix(y, y_pred)
clf_report = classification_report(y, y_pred)
f1_macro = f1_score(y, y_pred, average='weighted')

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:
print("Best Parameters:", grid.best_params_)
print("The f1_macro:", f1_macro)
print("The confusion Matrix:\n", cm)
print("The report:\n", clf_report)

Best Parameters: {'metric': 'euclidean', 'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
The f1_macro: 0.033556637806637805
The confusion Matrix:
 [[0 1 2 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [2 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
The report:
               precision    recall  f1-score   support

       15000       0.00      0.00      0.00         4
       16000       0.00      0.00      0.00         2
       17000       0.00      0.00      0.00         3
       18000       0.25      0.50      0.33         4
       19000       0.00      0.00      0.00         2
       20000       0.00      0.00      0.00         5
       21000       0.00      0.00      0.00         2
       22000       0.00      0.00      0.00         5
       23000       0.00      0.00      0.00         7
       25000       0.00      0.00      0.00         4
       26000       0.00      0.00      0.00         4
       27000       0.00      0.00      0.00         3
       28000    

In [39]:
import pickle
filename = "KNeighborsClassifier_CV_best_model.sav"
pickle.dump(grid.best_estimator_, open(filename, 'wb'))

In [40]:
results_df = pd.DataFrame(grid.cv_results_)
results_df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_p,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001985,0.000330,0.006898,0.001024,euclidean,3,1,uniform,"{'metric': 'euclidean', 'n_neighbors': 3, 'p':...",0.018750,0.012500,0.031250,0.018601,0.023750,0.020970,0.006256,37
1,0.001709,0.000072,0.009205,0.000607,euclidean,3,1,distance,"{'metric': 'euclidean', 'n_neighbors': 3, 'p':...",0.020833,0.014583,0.037500,0.018601,0.025000,0.023304,0.007857,25
2,0.001866,0.000413,0.006831,0.000698,euclidean,3,2,uniform,"{'metric': 'euclidean', 'n_neighbors': 3, 'p':...",0.018750,0.012500,0.031250,0.018601,0.023750,0.020970,0.006256,37
3,0.002137,0.000912,0.009646,0.000834,euclidean,3,2,distance,"{'metric': 'euclidean', 'n_neighbors': 3, 'p':...",0.020833,0.014583,0.037500,0.018601,0.025000,0.023304,0.007857,25
4,0.001909,0.000567,0.006626,0.000516,euclidean,5,1,uniform,"{'metric': 'euclidean', 'n_neighbors': 5, 'p':...",0.010417,0.025000,0.052083,0.008333,0.009821,0.021131,0.016610,34


In [41]:
# Get user input
age = int(input("Enter Age: "))
gender_male = int(input("Enter Gender (1 for Male, 0 for Female): "))
purchased = int(input("Enter Purchased (1 for Yes, 0 for No): "))
# Estimated Salary is the target variable, so it should not be used as input for prediction
# estimated_salary = int(input("Enter Estimated Salary: "))

# Create a numpy array from user input, matching the features used for training (Age, Gender_Male, Purchased)
user_input = np.array([[age, gender_male, purchased]])

# Scale the user input using the same scaler fitted on the training data
# The scaler was fitted on X which now only includes 'Age', 'Gender_Male', 'Purchased'
scaled_user_input = sc.transform(user_input)

# Make a prediction using the best estimator from the grid search
predicted_estimated_salary = grid.best_estimator_.predict(scaled_user_input)

print(f"The predicted Estimated Salary for the given input is: {predicted_estimated_salary[0]}")

Enter Age: 23
Enter Gender (1 for Male, 0 for Female): 1
Enter Purchased (1 for Yes, 0 for No): 1
The predicted Estimated Salary for the given input is: 123000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
